In [ ]:
import altair as alt
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.representation import (
    KeyBERTInspired,
    MaximalMarginalRelevance,
    # OpenAI,
    PartOfSpeech,
)
from hdbscan import HDBSCAN
import numpy as np
import pandas as pd
import plotly.express as px
import random
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import torch
from typing import List, Optional, Union
from umap import UMAP

from dsp_interview_transcripts import PROJECT_DIR, logger
from dsp_interview_transcripts.utils.viz import create_scatterplot
from dsp_interview_transcripts.utils.repr_docs import *

# Set random seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SENTENCE_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

alt.data_transformers.disable_max_rows()

pd.set_option("max_colwidth", 1000)

In [ ]:
df = pd.read_csv("s3://dsp-qualfml/interim/user_messages_min_len_9_w_sentiment.csv")

In [ ]:
empty_reduction_model = BaseDimensionalityReduction()

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = TfidfVectorizer(
    stop_words="english",
    min_df=1,
    max_df=0.85,
    ngram_range=(1, 3),
)

# KeyBERT
keybert_model = KeyBERTInspired()

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# All representation models
representation_model = {
                "KeyBERT": keybert_model,
                # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
                "MMR": mmr_model,
                # "POS": pos_model,
            }

topic_model = BERTopic(
                # Pipeline models
                embedding_model="sentence-transformers/all-MiniLM-L6-v2",
                umap_model=empty_reduction_model,
                hdbscan_model=hdbscan_model,
                vectorizer_model=vectorizer_model,
                representation_model=representation_model,
                # Hyperparameters
                top_n_words=10,
                verbose=True,
                calculate_probabilities=True,
            )
            
# Convert NaNs to empty strings
df['text_clean'] = df['text_clean'].astype(str)
docs = df['text_clean'].tolist()
embeddings = SENTENCE_MODEL.encode(docs, show_progress_bar=True)
normalized_embeddings = normalize(embeddings, norm='l2')
umap_2d = UMAP(random_state=RANDOM_SEED, n_components=2)
embeddings_2d = umap_2d.fit_transform(normalized_embeddings)

topics, probs = topic_model.fit_transform(docs, embeddings_2d)

rep_docs = topic_model.get_representative_docs()

topic_lookup = topic_model.get_topic_info()[["Topic", "Name"]]

df_vis = pd.DataFrame(embeddings_2d, columns=["x", "y"])
df_vis["topic"] = topics
df_vis = df_vis.merge(topic_lookup, left_on="topic", right_on="Topic", how="left")
df_vis["doc"] = docs

In [ ]:
df_vis = pd.merge(
                    df[["uuid","conversation",'text_clean', 'sentiment', 'question', 'context']],
                    df_vis,
                    left_on='text_clean',
                    right_on="doc",
                    how="outer",
                )

df_vis['Name'].value_counts(normalize=True)

In [ ]:
# Visualise clusters
fig = create_scatterplot(df_vis, color="Name:N", tooltip=["Name:N", "text_clean:N"])
fig

In [6]:
df_vis['norm_embedding'] = list(embeddings_2d)

In [7]:
topic_lookup = topic_model.get_topic_info()[["Topic", "Representation"]]

In [8]:
df_vis = (df_vis
          .merge(topic_lookup, left_on="topic", right_on="Topic", how="left")
          .drop(columns=['Topic_y'])
          .rename(columns={'Topic_x': 'Topic'}))

In [9]:
df_vis_no_noise = df_vis[df_vis['topic'] != -1]

In [10]:
radius_distributions, clustered_data = get_min_radius(df_vis_no_noise, k_neighbours=10)

In [11]:
clustered_data['quartile'] = clustered_data.groupby('topic')['radius_10'].transform(
    lambda x: pd.qcut(x, q=4, labels=['1st', '2nd', '3rd', 'greater than 3rd'])
)

In [ ]:
# Plot histogram for each cluster's radius distribution
for cluster_label, radii in radius_distributions.items():
    cluster_size = len(clustered_data[clustered_data['topic'] == cluster_label])
    fig = px.histogram(
        x=radii,
        nbins=20,
        title=f'Radius Distribution for Cluster {cluster_label} ({cluster_size} points in cluster)',
        labels={'x': f'Smallest Radius containing 10 Neighbors', 'y': 'Frequency'}
    )
    fig.update_layout(xaxis_title=f'Smallest Radius containing 10 Neighbors', yaxis_title='Frequency')
    fig.show()


In [ ]:
repr_docs = extract_repr_docs(clustered_data)

In [ ]:
# This plot unfortunately just doesn't seem informative
fig2 = create_scatterplot(clustered_data, color="quartile:N", tooltip=["Name:N", "text_clean:N"])
fig2

In [15]:
# Normalize 'radius_10' within each cluster
clustered_data['radius_10_zscore'] = clustered_data.groupby('Name')['radius_10'].transform(lambda x: abs(x - x.mean()) / x.std())


In [ ]:
clustered_data['radius_10_zscore'].hist()

In [ ]:
# This too isn't very helpful, even once we have removed outliers
fig = (
    alt.Chart(clustered_data[clustered_data['radius_10_zscore'] <3])
    .mark_circle(size=50)
    .encode(
        x=alt.X(
            "x:Q",
            axis=alt.Axis(ticks=False, labels=False, title=None, grid=False),
        ),
        y=alt.Y(
            "y:Q",
            axis=alt.Axis(ticks=False, labels=False, title=None, grid=False),
        ),
        color=alt.Color(
            "radius_10_zscore:Q",
            scale=alt.Scale(scheme="viridis"),
            legend=alt.Legend(title="Z-Score of Radius")
        ),
        shape=alt.Shape(
            "Name:N",
            legend=alt.Legend(title="Name")
        ),
        tooltip=["Name:N", "text_clean:N", "radius_10_zscore:Q"],
    )
    .properties(width=900, height=600)
    .interactive()
)

fig